In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
#Create spark session
spark = SparkSession.builder.appName("fiu_trans_moni").getOrCreate()

In [0]:
#Read CSV Input File
df = spark.read.format("csv")\
    .option("inferSchema", True)\
        .option("header", True)\
            .load("/Volumes/sam_catalog/sam_schema/fiu_trans/fiu_transactions.csv")
display(df)

transaction_id,customer_id,transaction_amount,transaction_type,country,transaction_date
1001,C101,25000,UPI,India,01-05-2026
1002,C102,150000,Wire Transfer,UAE,02-05-2026
1003,C101,75000,Cash Withdrawal,India,03-05-2026
1004,C103,220000,Wire Transfer,Singapore,03-05-2026
1005,C104,null,ATM Withdrawal,India,04-05-2026
1006,C102,300000,Cash Deposit,UAE,05-05-2026
1007,C105,120000,Online Transfer,null,05-05-2026
1008,C106,45000,UPI,India,06-05-2026
1009,C103,98000,Cash Withdrawal,Singapore,06-05-2026
1010,C101,175000,Wire Transfer,India,07-05-2026


In [0]:
#Remove duplicate data
df = df.dropDuplicates()
display(df)

transaction_id,customer_id,transaction_amount,transaction_type,country,transaction_date
1003,C101,75000,Cash Withdrawal,India,03-05-2026
1002,C102,150000,Wire Transfer,UAE,02-05-2026
1005,C104,null,ATM Withdrawal,India,04-05-2026
1004,C103,220000,Wire Transfer,Singapore,03-05-2026
1009,C103,98000,Cash Withdrawal,Singapore,06-05-2026
1007,C105,120000,Online Transfer,null,05-05-2026
1010,C101,175000,Wire Transfer,India,07-05-2026
1006,C102,300000,Cash Deposit,UAE,05-05-2026
1001,C101,25000,UPI,India,01-05-2026
1008,C106,45000,UPI,India,06-05-2026


In [0]:
# Handle Null Values
df = df.fillna({
    "country": "Unknown",
    "transaction_amount": 0
})
display(df)

transaction_id,customer_id,transaction_amount,transaction_type,country,transaction_date
1003,C101,75000,Cash Withdrawal,India,03-05-2026
1002,C102,150000,Wire Transfer,UAE,02-05-2026
1005,C104,0,ATM Withdrawal,India,04-05-2026
1004,C103,220000,Wire Transfer,Singapore,03-05-2026
1009,C103,98000,Cash Withdrawal,Singapore,06-05-2026
1007,C105,120000,Online Transfer,Unknown,05-05-2026
1010,C101,175000,Wire Transfer,India,07-05-2026
1006,C102,300000,Cash Deposit,UAE,05-05-2026
1001,C101,25000,UPI,India,01-05-2026
1008,C106,45000,UPI,India,06-05-2026


In [0]:
#Convert transaction date to proper date format
df = df.withColumn(
    "transaction_date",
    to_date(col("transaction_date"), "dd-MM-yyyy")
)
display(df)

transaction_id,customer_id,transaction_amount,transaction_type,country,transaction_date
1003,C101,75000,Cash Withdrawal,India,2026-05-03
1002,C102,150000,Wire Transfer,UAE,2026-05-02
1005,C104,0,ATM Withdrawal,India,2026-05-04
1004,C103,220000,Wire Transfer,Singapore,2026-05-03
1009,C103,98000,Cash Withdrawal,Singapore,2026-05-06
1007,C105,120000,Online Transfer,Unknown,2026-05-05
1010,C101,175000,Wire Transfer,India,2026-05-07
1006,C102,300000,Cash Deposit,UAE,2026-05-05
1001,C101,25000,UPI,India,2026-05-01
1008,C106,45000,UPI,India,2026-05-06


In [0]:
#Risk classifiction logic
df = df.withColumn(
    "risk_level",
    when(col("transaction_amount") > 200000, "HIGH")
    .when(
        (col("transaction_amount") >= 100000) & 
        (col("transaction_amount") <= 200000),
        "MEDIUM"
    )
    .otherwise("LOW")
)
display(df)

transaction_id,customer_id,transaction_amount,transaction_type,country,transaction_date,risk_level
1003,C101,75000,Cash Withdrawal,India,2026-05-03,LOW
1002,C102,150000,Wire Transfer,UAE,2026-05-02,MEDIUM
1005,C104,0,ATM Withdrawal,India,2026-05-04,LOW
1004,C103,220000,Wire Transfer,Singapore,2026-05-03,HIGH
1009,C103,98000,Cash Withdrawal,Singapore,2026-05-06,LOW
1007,C105,120000,Online Transfer,Unknown,2026-05-05,MEDIUM
1010,C101,175000,Wire Transfer,India,2026-05-07,MEDIUM
1006,C102,300000,Cash Deposit,UAE,2026-05-05,HIGH
1001,C101,25000,UPI,India,2026-05-01,LOW
1008,C106,45000,UPI,India,2026-05-06,LOW


In [0]:
# Detect Suspicious International Transactions
df = df.withColumn(
    "suspicious_flag",
    when(
        (col("transaction_amount") > 150000) &
        (col("country") != "India"),
        "YES"
    ).otherwise("NO")
)
display(df)

transaction_id,customer_id,transaction_amount,transaction_type,country,transaction_date,risk_level,suspicious_flag
1003,C101,75000,Cash Withdrawal,India,2026-05-03,LOW,NO
1002,C102,150000,Wire Transfer,UAE,2026-05-02,MEDIUM,NO
1005,C104,0,ATM Withdrawal,India,2026-05-04,LOW,NO
1004,C103,220000,Wire Transfer,Singapore,2026-05-03,HIGH,YES
1009,C103,98000,Cash Withdrawal,Singapore,2026-05-06,LOW,NO
1007,C105,120000,Online Transfer,Unknown,2026-05-05,MEDIUM,NO
1010,C101,175000,Wire Transfer,India,2026-05-07,MEDIUM,NO
1006,C102,300000,Cash Deposit,UAE,2026-05-05,HIGH,YES
1001,C101,25000,UPI,India,2026-05-01,LOW,NO
1008,C106,45000,UPI,India,2026-05-06,LOW,NO


In [0]:
#Generate customer level aggregation
cust_summary = df.groupBy("customer_id") \
    .agg(
        sum("transaction_amount").alias("total_transaction_amount"),
        round(avg("transaction_amount"), 2).alias("avg_transaction_amount"),
        count("transaction_id").alias("transaction_count")
    )
display(cust_summary)

customer_id,total_transaction_amount,avg_transaction_amount,transaction_count
C101,275000,91666.67,3
C102,450000,225000.0,2
C104,0,0.0,1
C103,318000,159000.0,2
C105,120000,120000.0,1
C106,45000,45000.0,1


In [0]:
# Define Window Specification
windowSpec = Window.partitionBy("country") \
    .orderBy(col("transaction_amount").desc())
display(windowSpec)

WindowSpec(PartitionBy(country), OrderBy(transaction_amount DESC NULLS LAST))

In [0]:
#Ranked transaction country wise
ranked_df = df.withColumn(
    "transaction_rank",
    dense_rank().over(windowSpec)        
)
display(ranked_df)

transaction_id,customer_id,transaction_amount,transaction_type,country,transaction_date,risk_level,suspicious_flag,transaction_rank
1010,C101,175000,Wire Transfer,India,2026-05-07,MEDIUM,NO,1
1003,C101,75000,Cash Withdrawal,India,2026-05-03,LOW,NO,2
1008,C106,45000,UPI,India,2026-05-06,LOW,NO,3
1001,C101,25000,UPI,India,2026-05-01,LOW,NO,4
1005,C104,0,ATM Withdrawal,India,2026-05-04,LOW,NO,5
1004,C103,220000,Wire Transfer,Singapore,2026-05-03,HIGH,YES,1
1009,C103,98000,Cash Withdrawal,Singapore,2026-05-06,LOW,NO,2
1006,C102,300000,Cash Deposit,UAE,2026-05-05,HIGH,YES,1
1002,C102,150000,Wire Transfer,UAE,2026-05-02,MEDIUM,NO,2
1007,C105,120000,Online Transfer,Unknown,2026-05-05,MEDIUM,NO,1


In [0]:
#Join Ranked Data with Customer Summary
final_df = ranked_df.join(cust_summary, on="customer_id", how = "left")
display(final_df)

customer_id,transaction_id,transaction_amount,transaction_type,country,transaction_date,risk_level,suspicious_flag,transaction_rank,total_transaction_amount,avg_transaction_amount,transaction_count
C101,1010,175000,Wire Transfer,India,2026-05-07,MEDIUM,NO,1,275000,91666.67,3
C101,1003,75000,Cash Withdrawal,India,2026-05-03,LOW,NO,2,275000,91666.67,3
C106,1008,45000,UPI,India,2026-05-06,LOW,NO,3,45000,45000.0,1
C101,1001,25000,UPI,India,2026-05-01,LOW,NO,4,275000,91666.67,3
C104,1005,0,ATM Withdrawal,India,2026-05-04,LOW,NO,5,0,0.0,1
C103,1004,220000,Wire Transfer,Singapore,2026-05-03,HIGH,YES,1,318000,159000.0,2
C103,1009,98000,Cash Withdrawal,Singapore,2026-05-06,LOW,NO,2,318000,159000.0,2
C102,1006,300000,Cash Deposit,UAE,2026-05-05,HIGH,YES,1,450000,225000.0,2
C102,1002,150000,Wire Transfer,UAE,2026-05-02,MEDIUM,NO,2,450000,225000.0,2
C105,1007,120000,Online Transfer,Unknown,2026-05-05,MEDIUM,NO,1,120000,120000.0,1


In [0]:
#Display Final Output
final_df.select(
    "customer_id","transaction_amount","country","risk_level","suspicious_flag","transaction_rank","total_transaction_amount",
    "avg_transaction_amount","transaction_count"
).display(truncate=False)

customer_id,transaction_amount,country,risk_level,suspicious_flag,transaction_rank,total_transaction_amount,avg_transaction_amount,transaction_count
C101,175000,India,MEDIUM,NO,1,275000,91666.67,3
C101,75000,India,LOW,NO,2,275000,91666.67,3
C106,45000,India,LOW,NO,3,45000,45000.0,1
C101,25000,India,LOW,NO,4,275000,91666.67,3
C104,0,India,LOW,NO,5,0,0.0,1
C103,220000,Singapore,HIGH,YES,1,318000,159000.0,2
C103,98000,Singapore,LOW,NO,2,318000,159000.0,2
C102,300000,UAE,HIGH,YES,1,450000,225000.0,2
C102,150000,UAE,MEDIUM,NO,2,450000,225000.0,2
C105,120000,Unknown,MEDIUM,NO,1,120000,120000.0,1


In [0]:
# Store Final Output as Delta Table for Querying
final_df.write.mode("overwrite") \
    .saveAsTable("sam_catalog.sam_schema.final_fiu_trans")

In [0]:
%sql
--Quering the final_fiu_trans data
SELECT * FROM sam_catalog.sam_schema.final_fiu_trans;

customer_id,transaction_id,transaction_amount,transaction_type,country,transaction_date,risk_level,suspicious_flag,transaction_rank,total_transaction_amount,avg_transaction_amount,transaction_count
C101,1010,175000,Wire Transfer,India,2026-05-07,MEDIUM,NO,1,275000,91666.67,3
C101,1003,75000,Cash Withdrawal,India,2026-05-03,LOW,NO,2,275000,91666.67,3
C106,1008,45000,UPI,India,2026-05-06,LOW,NO,3,45000,45000.0,1
C101,1001,25000,UPI,India,2026-05-01,LOW,NO,4,275000,91666.67,3
C104,1005,0,ATM Withdrawal,India,2026-05-04,LOW,NO,5,0,0.0,1
C103,1004,220000,Wire Transfer,Singapore,2026-05-03,HIGH,YES,1,318000,159000.0,2
C103,1009,98000,Cash Withdrawal,Singapore,2026-05-06,LOW,NO,2,318000,159000.0,2
C102,1006,300000,Cash Deposit,UAE,2026-05-05,HIGH,YES,1,450000,225000.0,2
C102,1002,150000,Wire Transfer,UAE,2026-05-02,MEDIUM,NO,2,450000,225000.0,2
C105,1007,120000,Online Transfer,Unknown,2026-05-05,MEDIUM,NO,1,120000,120000.0,1
